# MoNuSAC Download and Extraction

This notebook loads `RationAI/MoNuSAC`, merges the `train` and `test` splits, and exports paired image and mask PNG files under `../../data/Monusac/`.

Assumptions:
- The notebook is saved and executed from `scripts/benchmarking/`.
- If Hugging Face asks for authentication, run `huggingface-cli login` or `hf auth login` in your shell first.
- Each exported `*_mask.png` is a single merged instance mask where background is `0` and each nucleus gets its own positive integer ID.


In [3]:
from __future__ import annotations

import io
import os
import re
import shutil
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image
from datasets import concatenate_datasets, load_dataset

NOTEBOOK_DIR = Path("/share/lab_teng/trainee/tusharsingh/cell-seg/scripts/benchmarking").resolve()
PROJECT_ROOT = (NOTEBOOK_DIR / "/../../").resolve()
DATA_ROOT = (NOTEBOOK_DIR / "../../data/Monusac").resolve()
ALL_MERGED_DIR = DATA_ROOT / "all_merged"
KIDNEY_ONLY_DIR = DATA_ROOT / "kidney_only"

print(f"Current working directory: {os.getcwd()}")
print(f"Notebook directory assumption: {NOTEBOOK_DIR}")
print(f"Project root resolved from ../../: {PROJECT_ROOT}")
print(f"Data root resolved from ../../data/Monusac: {DATA_ROOT}")

for directory in (DATA_ROOT, ALL_MERGED_DIR, KIDNEY_ONLY_DIR):
    directory.mkdir(parents=True, exist_ok=True)
    print(f"Ensured directory exists: {directory}")


Current working directory: /share/lab_teng/trainee/tusharsingh/cell-seg
Notebook directory assumption: /share/lab_teng/trainee/tusharsingh/cell-seg/scripts/benchmarking
Project root resolved from ../../: /
Data root resolved from ../../data/Monusac: /share/lab_teng/trainee/tusharsingh/cell-seg/data/Monusac
Ensured directory exists: /share/lab_teng/trainee/tusharsingh/cell-seg/data/Monusac
Ensured directory exists: /share/lab_teng/trainee/tusharsingh/cell-seg/data/Monusac/all_merged
Ensured directory exists: /share/lab_teng/trainee/tusharsingh/cell-seg/data/Monusac/kidney_only


In [4]:
print("Loading MoNuSAC from Hugging Face...")
raw_ds = load_dataset("RationAI/MoNuSAC")
print(f"Loaded splits: {list(raw_ds.keys())}")

train_ds = raw_ds["train"].add_column("source_split", ["train"] * len(raw_ds["train"]))
train_ds = train_ds.add_column("source_index", list(range(len(raw_ds["train"]))))
test_ds = raw_ds["test"].add_column("source_split", ["test"] * len(raw_ds["test"]))
test_ds = test_ds.add_column("source_index", list(range(len(raw_ds["test"]))))

merged_ds = concatenate_datasets([train_ds, test_ds])

tissue_feature = raw_ds["train"].features["tissue"]
category_feature = raw_ds["train"].features["categories"].feature
kidney_label = tissue_feature.str2int("Kidney")

def tissue_to_name(value: int | str) -> str:
    if isinstance(value, str):
        return value
    return tissue_feature.int2str(int(value))

def category_to_name(value: int | str) -> str:
    if isinstance(value, str):
        return value
    return category_feature.int2str(int(value))

tissue_counts = Counter(tissue_to_name(value) for value in merged_ds["tissue"])

print(f"Train samples: {len(train_ds)}")
print(f"Test samples: {len(test_ds)}")
print(f"Merged samples: {len(merged_ds)}")
print("Tissue distribution:")
for tissue_name, count in sorted(tissue_counts.items()):
    print(f"  - {tissue_name}: {count}")

sample_row = merged_ds[0]
print("\nFirst sample preview:")
print(f"  patient: {sample_row['patient']}")
print(f"  source split/index: {sample_row['source_split']} / {sample_row['source_index']}")
print(f"  tissue: {tissue_to_name(sample_row['tissue'])}")
print(f"  number of instance masks: {len(sample_row['instances'])}")
print("  category labels present:", [category_to_name(value) for value in sample_row["categories"][:5]])


Loading MoNuSAC from Hugging Face...


Generating train split:   0%|          | 0/209 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/101 [00:00<?, ? examples/s]

Loaded splits: ['train', 'test']
Train samples: 209
Test samples: 101
Merged samples: 310
Tissue distribution:
  - Breast: 80
  - Kidney: 86
  - Lung: 61
  - Prostate: 83

First sample preview:
  patient: TCGA-73-4668
  source split/index: train / 0
  tissue: Lung
  number of instance masks: 17
  category labels present: ['Lymphocyte', 'Lymphocyte', 'Lymphocyte', 'Lymphocyte', 'Lymphocyte']


In [5]:
def safe_stem(value: str) -> str:
    cleaned = re.sub(r"[^A-Za-z0-9._-]+", "_", value.strip())
    return cleaned.strip("._") or "sample"

def ensure_pil_image(image_like, mode: str | None = None) -> Image.Image:
    if isinstance(image_like, Image.Image):
        image = image_like.copy()
    elif isinstance(image_like, np.ndarray):
        image = Image.fromarray(image_like)
    elif isinstance(image_like, dict):
        if image_like.get("bytes") is not None:
            with Image.open(io.BytesIO(image_like["bytes"])) as pil_image:
                image = pil_image.copy()
        elif image_like.get("path"):
            with Image.open(image_like["path"]) as pil_image:
                image = pil_image.copy()
        else:
            raise ValueError("Unsupported image dictionary without bytes or path.")
    else:
        raise TypeError(f"Unsupported image type: {type(image_like)!r}")

    if mode is not None:
        image = image.convert(mode)

    return image

def binary_mask_array(mask_like) -> np.ndarray:
    mask_image = ensure_pil_image(mask_like)
    mask_array = np.asarray(mask_image)
    if mask_array.ndim == 3:
        mask_array = mask_array[..., 0]
    return mask_array > 0

def build_instance_mask(instance_masks, image_size: tuple[int, int]) -> tuple[np.ndarray, int]:
    width, height = image_size
    merged_mask = np.zeros((height, width), dtype=np.uint16)
    overlap_pixels = 0

    for instance_id, instance_mask in enumerate(instance_masks, start=1):
        if instance_id > np.iinfo(np.uint16).max:
            raise ValueError("Too many instances for a uint16 PNG mask.")

        current_mask = binary_mask_array(instance_mask)
        if current_mask.shape != merged_mask.shape:
            raise ValueError(
                f"Instance mask shape mismatch: expected {merged_mask.shape}, got {current_mask.shape}"
            )

        overlap_pixels += int(np.count_nonzero((merged_mask > 0) & current_mask))
        merged_mask[current_mask] = instance_id

    return merged_mask, overlap_pixels

def is_kidney_value(value: int | str) -> bool:
    return value.lower() == "kidney" if isinstance(value, str) else int(value) == kidney_label

def save_sample(row, merged_index: int, destination: Path) -> dict[str, object]:
    tissue_name = tissue_to_name(row["tissue"])
    patient_id = safe_stem(str(row["patient"]))
    unique_id = (
        f"{merged_index:04d}_{row['source_split']}_{int(row['source_index']):04d}_"
        f"{patient_id}_{safe_stem(tissue_name.lower())}"
    )

    rgb_image = ensure_pil_image(row["image"], mode="RGB")
    merged_mask, overlap_pixels = build_instance_mask(row["instances"], rgb_image.size)

    image_path = destination / f"{unique_id}_image.png"
    mask_path = destination / f"{unique_id}_mask.png"

    rgb_image.save(image_path, format="PNG")
    Image.fromarray(merged_mask).save(mask_path, format="PNG")

    return {
        "unique_id": unique_id,
        "patient": row["patient"],
        "tissue": tissue_name,
        "source_split": row["source_split"],
        "source_index": int(row["source_index"]),
        "num_instances": len(row["instances"]),
        "overlap_pixels": overlap_pixels,
        "image_path": str(image_path),
        "mask_path": str(mask_path),
    }

print("Helper functions ready.")


Helper functions ready.


In [6]:
print("Starting export to ../../data/Monusac ...")
records = []
kidney_exports = 0

for merged_index, row in enumerate(merged_ds):
    record = save_sample(row, merged_index=merged_index, destination=ALL_MERGED_DIR)
    records.append(record)

    if is_kidney_value(row["tissue"]):
        kidney_image_target = KIDNEY_ONLY_DIR / Path(record["image_path"]).name
        kidney_mask_target = KIDNEY_ONLY_DIR / Path(record["mask_path"]).name
        shutil.copy2(record["image_path"], kidney_image_target)
        shutil.copy2(record["mask_path"], kidney_mask_target)
        kidney_exports += 1

    if (merged_index + 1) % 25 == 0 or (merged_index + 1) == len(merged_ds):
        print(
            f"Processed {merged_index + 1}/{len(merged_ds)} samples | "
            f"Kidney copies written: {kidney_exports}"
        )

summary_df = pd.DataFrame(records)
summary_path = DATA_ROOT / "extraction_summary.csv"
summary_df.to_csv(summary_path, index=False)

print("\nExtraction complete.")
print(f"All merged directory: {ALL_MERGED_DIR}")
print(f"Kidney-only directory: {KIDNEY_ONLY_DIR}")
print(f"Summary CSV: {summary_path}")
print(f"Total exported samples: {len(summary_df)}")
print(f"Kidney-only exported samples: {kidney_exports}")


Starting export to ../../data/Monusac ...
Processed 25/310 samples | Kidney copies written: 6
Processed 50/310 samples | Kidney copies written: 15
Processed 75/310 samples | Kidney copies written: 25
Processed 100/310 samples | Kidney copies written: 25
Processed 125/310 samples | Kidney copies written: 41
Processed 150/310 samples | Kidney copies written: 43
Processed 175/310 samples | Kidney copies written: 48
Processed 200/310 samples | Kidney copies written: 55
Processed 225/310 samples | Kidney copies written: 59
Processed 250/310 samples | Kidney copies written: 70
Processed 275/310 samples | Kidney copies written: 80
Processed 300/310 samples | Kidney copies written: 86
Processed 310/310 samples | Kidney copies written: 86

Extraction complete.
All merged directory: /share/lab_teng/trainee/tusharsingh/cell-seg/data/Monusac/all_merged
Kidney-only directory: /share/lab_teng/trainee/tusharsingh/cell-seg/data/Monusac/kidney_only
Summary CSV: /share/lab_teng/trainee/tusharsingh/cell-

In [7]:
disk_summary = pd.DataFrame(
    {
        "folder": ["all_merged", "kidney_only"],
        "image_files": [
            len(list(ALL_MERGED_DIR.glob("*_image.png"))),
            len(list(KIDNEY_ONLY_DIR.glob("*_image.png"))),
        ],
        "mask_files": [
            len(list(ALL_MERGED_DIR.glob("*_mask.png"))),
            len(list(KIDNEY_ONLY_DIR.glob("*_mask.png"))),
        ],
    }
)

print("On-disk file counts:")
print(disk_summary.to_string(index=False))

print("\nFirst five kidney exports:")
print(summary_df.loc[summary_df["tissue"].eq("Kidney"), ["unique_id", "image_path", "mask_path"]].head().to_string(index=False))

summary_df.head()


On-disk file counts:
     folder  image_files  mask_files
 all_merged          310         310
kidney_only           86          86

First five kidney exports:
                          unique_id                                                                                                         image_path                                                                                                         mask_path
0009_train_0009_TCGA-EV-5903_kidney /share/lab_teng/trainee/tusharsingh/cell-seg/data/Monusac/all_merged/0009_train_0009_TCGA-EV-5903_kidney_image.png /share/lab_teng/trainee/tusharsingh/cell-seg/data/Monusac/all_merged/0009_train_0009_TCGA-EV-5903_kidney_mask.png
0010_train_0010_TCGA-EV-5903_kidney /share/lab_teng/trainee/tusharsingh/cell-seg/data/Monusac/all_merged/0010_train_0010_TCGA-EV-5903_kidney_image.png /share/lab_teng/trainee/tusharsingh/cell-seg/data/Monusac/all_merged/0010_train_0010_TCGA-EV-5903_kidney_mask.png
0011_train_0011_TCGA-EV-5903_kidney /share/la

,unique_id,patient,tissue,source_split,source_index,num_instances,overlap_pixels,image_path,mask_path
0,0000_train_0000_TCGA-73-4668_lung,TCGA-73-4668,Lung,train,0,17,0,/share/lab_teng/trainee/tusharsingh/cell-seg/d...,/share/lab_teng/trainee/tusharsingh/cell-seg/d...
1,0001_train_0001_TCGA-73-4668_lung,TCGA-73-4668,Lung,train,1,5,0,/share/lab_teng/trainee/tusharsingh/cell-seg/d...,/share/lab_teng/trainee/tusharsingh/cell-seg/d...
2,0002_train_0002_TCGA-73-4668_lung,TCGA-73-4668,Lung,train,2,302,1724,/share/lab_teng/trainee/tusharsingh/cell-seg/d...,/share/lab_teng/trainee/tusharsingh/cell-seg/d...
3,0003_train_0003_TCGA-73-4668_lung,TCGA-73-4668,Lung,train,3,203,62,/share/lab_teng/trainee/tusharsingh/cell-seg/d...,/share/lab_teng/trainee/tusharsingh/cell-seg/d...
4,0004_train_0004_TCGA-55-1594_lung,TCGA-55-1594,Lung,train,4,251,608,/share/lab_teng/trainee/tusharsingh/cell-seg/d...,/share/lab_teng/trainee/tusharsingh/cell-seg/d...
